# FinPipeline Comparison — MiniLM vs BGE-Large × Dense vs Hybrid
**Goal:** Compare 4 end-to-end retrieval pipelines on the same reduced corpus and query sample.

| Pipeline | Dense Model       | Retrieval      | QE  | Reranker      | Answer |
|----------|-------------------|----------------|-----|---------------|--------|
| **A1**   | MiniLM (384-dim)  | Dense only     | Yes | Mistral Large | Groq   |
| **A2**   | MiniLM (384-dim)  | Hybrid α=0.7   | Yes | Mistral Large | Groq   |
| **B1**   | BGE-Large (1024-dim) | Dense only  | Yes | Mistral Large | Groq   |
| **B2**   | BGE-Large (1024-dim) | Hybrid α=0.7| Yes | Mistral Large | Groq   |

**Controlled variables (same for all 4):**
- Stratified corpus: all qrel docs + 8% random negatives (~6K passages)
- Query sample: 30% of FiQA test queries, `random_state=42`
- Query expansion: Groq Llama 3.3 70B (text concat before encoding)
- Reranker: `mistralai/mistral-large-2411` via OpenRouter
- Answer: `meta-llama/llama-3.3-70b-instruct` via OpenRouter
- NLI faithfulness: `cross-encoder/nli-deberta-v3-small` (entailment index=2)
- Results saved to: `finrerank/Pipeline_Results/`

## Cell 0 — Repo Root & Config

In [2]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent

os.chdir(_root)
sys.path.insert(0, str(_root))

from config import FIQA_CORPUS, FIQA_QUERIES, FIQA_QRELS_TEST

PIPELINE_DIR = os.path.join('finrerank', 'Pipeline_Results')
os.makedirs(PIPELINE_DIR, exist_ok=True)

print('Repo root   :', _root)
print('Results dir :', PIPELINE_DIR)

Repo root   : /Users/momo/Desktop/GEN AI/Finsearch Project/FinSearch_Intent_Aware_Financial_Document_Intelligence-
Results dir : finrerank/Pipeline_Results


## Cell 1 — Install Dependencies

In [3]:
import sys
!{sys.executable} -m pip install -q sentence-transformers faiss-cpu rank_bm25 pytrec-eval-terrier openai python-dotenv

## Cell 2 — Imports

In [4]:
import os, json, re, time, gc, pickle
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import numpy as np
import pandas as pd
import faiss
import pytrec_eval
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY not found in .env file')

print('Imports OK')
print('faiss version :', faiss.__version__)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
faiss version : 1.13.2


## Cell 3 — Load FiQA Corpus, Queries & Qrels
Applies same stratified corpus reduction and 30% query sample used in `FinDomain_ModelComparison.ipynb`.
- **Corpus**: all qrel docs + 8% random negatives — ensures every relevant doc is present
- **Queries**: 30% sample with `random_state=42` — identical queries across all 4 pipelines

In [5]:
print('Loading corpus...')
corpus_df = pd.read_csv(FIQA_CORPUS, usecols=['_id', 'text'])
corpus_df = corpus_df.dropna(subset=['text']).reset_index(drop=True)
corpus_df['_id'] = corpus_df['_id'].astype(str)
print('Full corpus:', len(corpus_df), 'passages')

queries_df = pd.read_csv(FIQA_QUERIES, usecols=['_id', 'text'])
queries_df['_id'] = queries_df['_id'].astype(str)
query_id_to_text = dict(zip(queries_df['_id'], queries_df['text']))

qrels_df = pd.read_csv(FIQA_QRELS_TEST)
qrels_df.columns = [c.strip() for c in qrels_df.columns]
qrels_df = qrels_df.astype(str)
query_col  = [c for c in qrels_df.columns if 'query'  in c.lower()][0]
corpus_col = [c for c in qrels_df.columns if 'corpus' in c.lower()][0]
score_col  = [c for c in qrels_df.columns if 'score'  in c.lower()][0]
qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row[query_col])
    did = str(row[corpus_col])
    qrels_dict.setdefault(qid, {})[did] = int(float(row[score_col]))

test_qids       = list(qrels_dict.keys())
test_queries_df = queries_df[queries_df['_id'].isin(test_qids)].reset_index(drop=True)
print('Test queries:', len(test_queries_df), '| Qrels:', len(qrels_dict))

# ── Stratified corpus reduction (same as FinDomain_ModelComparison.ipynb) ────
qrel_doc_ids = set(did for docs in qrels_dict.values() for did in docs.keys())
relevant_df  = corpus_df[corpus_df['_id'].isin(qrel_doc_ids)].reset_index(drop=True)
remaining_df = corpus_df[~corpus_df['_id'].isin(qrel_doc_ids)]
sampled_neg  = remaining_df.sample(frac=0.08, random_state=42).reset_index(drop=True)
corpus_df    = pd.concat([relevant_df, sampled_neg]).reset_index(drop=True)

corpus_ids        = corpus_df['_id'].tolist()
corpus_texts      = corpus_df['text'].tolist()
corpus_id_to_text = dict(zip(corpus_ids, corpus_texts))

print('\nReduced corpus  :', len(corpus_df), 'passages')
print('  Relevant docs :', len(relevant_df), '(all qrel docs kept)')
print('  Negatives     :', len(sampled_neg), '(8% random)')

# ── 30% query sample — same random_state=42 as FinDomain_ModelComparison ─────
sample_df    = test_queries_df.sample(frac=0.30, random_state=42).reset_index(drop=True)
sample_qids  = sample_df['_id'].tolist()
sample_texts = sample_df['text'].tolist()
sample_qrels = {qid: qrels_dict[qid] for qid in sample_qids if qid in qrels_dict}
print('\n30% query sample:', len(sample_df), 'queries (random_state=42, same as ModelComparison)')

Loading corpus...
Full corpus: 57600 passages
Test queries: 648 | Qrels: 648

Reduced corpus  : 6177 passages
  Relevant docs : 1705 (all qrel docs kept)
  Negatives     : 4472 (8% random)

30% query sample: 194 queries (random_state=42, same as ModelComparison)


## Cell 4 — Shared Helper Functions

In [6]:
def tokenize(text):
    t = str(text).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return t.split()


def evaluate_metrics(results_dict, qrels):
    """Compute NDCG@10, MRR, Recall@10 using pytrec_eval."""
    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels, {'ndcg_cut_10', 'recip_rank', 'recall_10'}
    )
    res = evaluator.evaluate(results_dict)
    return {
        'NDCG@10':     round(np.mean([v['ndcg_cut_10'] for v in res.values()]), 4),
        'MRR':         round(np.mean([v['recip_rank']  for v in res.values()]), 4),
        'Recall@10':   round(np.mean([v['recall_10']   for v in res.values()]), 4),
        'num_queries': len(res),
    }


def dense_retrieve(query_embs, faiss_index, ids, query_ids, top_k=50):
    """Search FAISS and return {qid: {doc_id: score}}."""
    results = {}
    D, I = faiss_index.search(query_embs, top_k)
    for i, qid in enumerate(query_ids):
        results[qid] = {
            ids[idx]: float(D[i, rank])
            for rank, idx in enumerate(I[i])
            if idx != -1 and idx < len(ids)
        }
    return results


def hybrid_retrieve(query_embs, faiss_index, ids, bm25_obj,
                    combined_texts_list, query_ids, alpha=0.7, top_k=50):
    """Alpha-fusion of dense + BM25. Returns top_k candidates per query."""
    results = {}
    D, I = faiss_index.search(query_embs, top_k * 2)
    for i, qid in enumerate(query_ids):
        # Dense scores
        dense_scores = {
            ids[idx]: float(D[i, rank])
            for rank, idx in enumerate(I[i])
            if idx != -1 and idx < len(ids)
        }
        # BM25 scores — tokenize the combined (original+expanded) query text
        toks     = tokenize(combined_texts_list[i])
        bm25_raw = bm25_obj.get_scores(toks)
        top_bm25 = np.argsort(bm25_raw)[::-1][:top_k * 2]
        bm25_scores = {ids[idx]: float(bm25_raw[idx]) for idx in top_bm25}

        # Alpha fusion: alpha * dense_norm + (1-alpha) * bm25_norm
        all_docs = set(dense_scores) | set(bm25_scores)
        d_vals   = np.array([dense_scores.get(d, 0.0) for d in all_docs])
        b_vals   = np.array([bm25_scores.get(d,  0.0) for d in all_docs])
        d_norm   = (d_vals - d_vals.min()) / (d_vals.max() - d_vals.min() + 1e-9)
        b_norm   = (b_vals - b_vals.min()) / (b_vals.max() - b_vals.min() + 1e-9)
        fused    = alpha * d_norm + (1 - alpha) * b_norm
        ranked   = sorted(zip(all_docs, fused), key=lambda x: x[1], reverse=True)[:top_k]
        results[qid] = {doc_id: float(score) for doc_id, score in ranked}
    return results


def mistral_rerank(query, candidates, corpus_id_to_text, client, top_k=10):
    """Re-rank candidates with Mistral Large. Sends top-20 to LLM (cost control)."""
    llm_cands = candidates[:20]
    lines = []
    for i, d in enumerate(llm_cands, 1):
        text = corpus_id_to_text.get(d, '')[:400].replace('\n', ' ')
        lines.append(str(i) + '. ' + text)
    n        = len(llm_cands)
    passages = '\n'.join(lines)
    prompt = (
        'You are a financial document relevance expert specializing in investment, '
        'tax, and personal finance.\n\n'
        'Query: "' + query + '"\n\n'
        'Below are ' + str(n) + ' candidate financial passages. '
        'Return a JSON array of passage numbers (1-based) sorted MOST to LEAST relevant. '
        'Include ALL ' + str(n) + ' numbers.\n\n'
        'Passages:\n' + passages + '\n\n'
        'Respond ONLY with a JSON array, e.g.: [3,1,5,2,4]'
    )
    try:
        r = client.chat.completions.create(
            model=MISTRAL_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=200, temperature=0,
        )
        raw = r.choices[0].message.content.strip()
        m   = re.search(r'\[[\d,\s]+\]', raw)
        if m:
            ranking = json.loads(m.group())
            ranking = [x for x in ranking if 1 <= x <= n]
            seen    = set(ranking)
            for j in range(1, n + 1):
                if j not in seen:
                    ranking.append(j)
            reranked = [llm_cands[idx - 1] for idx in ranking[:top_k]]
            seen_ids = set(reranked)
            for d in candidates[20:]:
                if len(reranked) >= top_k:
                    break
                if d not in seen_ids:
                    reranked.append(d)
            return {d: float(top_k - pos) for pos, d in enumerate(reranked[:top_k])}
    except Exception as e:
        print('  Mistral error:', e)
    return {d: float(n - i) for i, d in enumerate(candidates[:top_k])}


def groq_answer(query, top_docs, corpus_id_to_text, client):
    """Generate financial answer using Groq Llama 3.3 70B."""
    parts = []
    for i, d in enumerate(top_docs, 1):
        parts.append('[Doc ' + str(i) + '] ' + corpus_id_to_text.get(d, '')[:500])
    context = '\n'.join(parts)
    prompt = (
        'You are an expert financial advisor assistant.\n'
        'Answer the question using ONLY the provided documents.\n'
        'If the documents do not contain enough information, say so.\n\n'
        'Question: ' + query + '\n\n'
        'Documents:\n' + context + '\n\n'
        'Provide a clear, accurate financial answer based strictly on the documents above.'
    )
    try:
        r = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=300, temperature=0.1,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return 'Error: ' + str(e)


def compute_confidence(answer, top_docs, corpus_id_to_text, retrieval_scores, nli_model):
    """
    NLI-based faithfulness scoring.
    cross-encoder/nli-deberta-v3-small label order:
      index 0 = contradiction, index 1 = neutral, index 2 = entailment
    """
    scores = [retrieval_scores.get(d, 0.0) for d in top_docs]
    s_min, s_max = min(scores), max(scores)
    norm = [(s - s_min) / (s_max - s_min + 1e-9) for s in scores]
    retrieval_conf = float(np.mean(norm))

    entailment_scores = []
    for doc_id in top_docs:
        doc_text = corpus_id_to_text.get(doc_id, '')[:500]
        if not doc_text.strip():
            continue
        logits = nli_model.predict([[doc_text, answer]])
        probs  = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
        entailment_scores.append(float(probs[0][2]))  # index 2 = entailment
    faithfulness_conf = float(np.mean(entailment_scores)) if entailment_scores else 0.0

    final_conf = round(0.4 * retrieval_conf + 0.6 * faithfulness_conf, 4)
    return {
        'retrieval_confidence':    round(retrieval_conf,    4),
        'faithfulness_confidence': round(faithfulness_conf, 4),
        'final_confidence':        final_conf,
    }


print('Helper functions defined.')

Helper functions defined.


## Cell 5 — LLM Setup + Query Expansion (computed once, shared by all 4 pipelines)
Query expansion text is model-agnostic — same Groq-generated expansions reused for MiniLM and BGE-Large.  
The *encoding* of expanded text differs per model (MiniLM: no prefix | BGE-Large: query prefix).

In [7]:
GROQ_MODEL    = 'meta-llama/llama-3.3-70b-instruct'
MISTRAL_MODEL = 'mistralai/mistral-large-2411'
RATE_DELAY    = 2.5   # seconds between Mistral rerank calls
TOP_K         = 50    # retrieval candidates per query (Mistral sees top-20 of these)
ALPHA         = 0.7   # BM25 alpha for hybrid pipelines

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
)

# Smoke test
resp = client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{'role': 'user', 'content': 'Reply with one word: ready'}],
    max_tokens=5,
)
print('OpenRouter ready:', resp.choices[0].message.content.strip())


def query_expand(query, client, model=GROQ_MODEL):
    """
    Expand a financial query with LLM-generated synonyms/related terms.
    Returns: expanded string (original + finance terms on new line)
    """
    prompt = (
        'You are a financial search expert.\n'
        'Given the query below, add 8-12 relevant financial terms, synonyms, '
        'and related concepts that would help retrieve relevant documents.\n'
        'Output ONLY the original query followed by the extra terms on a new line. '
        'No explanation, no bullets, just terms separated by spaces.\n\n'
        'Query: ' + query
    )
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=80, temperature=0,
        )
        expanded = r.choices[0].message.content.strip()
        return expanded if expanded else query
    except Exception as e:
        print('  QE error:', e)
        return query


# ── Compute QE once for all 30% sample queries ───────────────────────────────
print('\nExpanding', len(sample_texts), 'queries (once — shared by all 4 pipelines)...')
expanded_queries = []
for qt in tqdm(sample_texts):
    expanded_queries.append(query_expand(qt, client))
    time.sleep(0.3)

# Text concatenation: original query + expanded terms (RIGHT method — not embedding averaging)
# Encoding once as single string preserves full context vs averaging separate vectors
combined_texts = [orig + ' ' + exp for orig, exp in zip(sample_texts, expanded_queries)]

print('Query expansion done.')
print('Example:')
print('  Original :', sample_texts[0])
print('  Combined :', combined_texts[0][:120], '...')

OpenRouter ready: Ready

Expanding 194 queries (once — shared by all 4 pipelines)...


100%|██████████| 194/194 [08:05<00:00,  2.50s/it]

Query expansion done.
Example:
  Original : What is the easiest way to back-test index funds and ETFs?
  Combined : What is the easiest way to back-test index funds and ETFs? What is the easiest way to back-test index funds and ETFs?
ba ...


## Cell 6 — Load NLI Model

In [8]:
print('Loading NLI confidence model: cross-encoder/nli-deberta-v3-small ...')
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-small',
                         default_activation_function=None)
print('NLI model loaded.')
print('Label order: [0=contradiction, 1=neutral, 2=entailment]  — using index 2')

The CrossEncoder `default_activation_function` argument was renamed and is now deprecated, please use `activation_fn` instead.


Loading NLI confidence model: cross-encoder/nli-deberta-v3-small ...


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 9989.80it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI model loaded.
Label order: [0=contradiction, 1=neutral, 2=entailment]  — using index 2


## Cell 7 — Build BM25 (shared by Hybrid pipelines A2 and B2)
BM25 is lexical (no embeddings) so it is built once on the reduced corpus and reused for both MiniLM and BGE-Large hybrid pipelines.
Finance-tuned params: `k1=1.2, b=0.5`

In [10]:
BM25_K1 = 1.2  # lower than default 1.5 — finance jargon repeats often
BM25_B  = 0.5  # lower than default 0.75 — finance docs vary greatly in length

print('Building BM25 index (k1={}, b={}) ...'.format(BM25_K1, BM25_B))
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts)]
bm25 = BM25Okapi(tokenized_corpus, k1=BM25_K1, b=BM25_B)
print('BM25 ready ({:,} passages tokenized)'.format(len(tokenized_corpus)))

Building BM25 index (k1=1.2, b=0.5) ...


100%|██████████| 6177/6177 [00:00<00:00, 58725.75it/s]


BM25 ready (6,177 passages tokenized)


## Cell 8 — MiniLM: Encode Corpus + Build FAISS
`all-MiniLM-L6-v2` — 384-dim, 22M params, no query prefix needed.
After encoding, compute query embeddings from combined (original+expanded) texts.

In [15]:
print('=' * 55)
print('Encoding corpus with MiniLM...')
print('=' * 55)

minilm = SentenceTransformer('all-MiniLM-L6-v2')
print('  Dim:', minilm.get_sentence_embedding_dimension(), ' Params: ~22M')

# Encode corpus (no prefix for MiniLM)
minilm_corpus_embs = minilm.encode(
    corpus_texts,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)

dim_m = minilm_corpus_embs.shape[1]
minilm_index = faiss.IndexFlatIP(dim_m)
minilm_index.add(minilm_corpus_embs)
print('  FAISS index: dim={}  vectors={:,}'.format(dim_m, minilm_index.ntotal))

# Encode queries — use combined (original + expanded) text, no prefix
print('Encoding 30% sample queries with MiniLM + QE...')
minilm_query_embs = minilm.encode(
    combined_texts,    # text-concat QE (correct method)
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)

print('MiniLM ready. Query embs shape:', minilm_query_embs.shape)

Encoding corpus with MiniLM...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9310.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 384  Params: ~22M


Batches: 100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


  FAISS index: dim=384  vectors=6,177
Encoding 30% sample queries with MiniLM + QE...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]

MiniLM ready. Query embs shape: (194, 384)


## Cell 9 — Pipeline A1: MiniLM Dense + QE + Mistral Rerank
**Dense only** — no BM25 fusion. FAISS cosine search → top-50 → Mistral reranks top-20 → top-10.

In [16]:
print('=' * 60)
print('Pipeline A1: MiniLM Dense + QE + Mistral Rerank')
print('=' * 60)

# Step 1: Dense retrieval
print('Step 1: Dense retrieval (top-{})...'.format(TOP_K))
a1_retrieved = dense_retrieve(minilm_query_embs, minilm_index, corpus_ids, sample_qids, top_k=TOP_K)

# Recall@50 — ceiling for what reranker can achieve
eval_ret = pytrec_eval.RelevanceEvaluator(sample_qrels, {'recall_50'})
a1_recall50 = round(np.mean([v['recall_50'] for v in eval_ret.evaluate(a1_retrieved).values()]), 4)
print('  Recall@50 (retrieval ceiling):', a1_recall50)

# Step 2: Mistral Large re-ranking
print('Step 2: Mistral Large re-ranking ({} queries)...'.format(len(sample_qids)))
a1_reranked = {}
for i, (qid, q_text) in enumerate(tqdm(zip(sample_qids, sample_texts), total=len(sample_qids))):
    candidates = list(a1_retrieved[qid].keys())
    a1_reranked[qid] = mistral_rerank(q_text, candidates, corpus_id_to_text, client, top_k=10)
    time.sleep(RATE_DELAY)

# Step 3: Evaluate
a1_metrics = evaluate_metrics(a1_reranked, sample_qrels)
print('\nPipeline A1 Results ({} queries):'.format(a1_metrics['num_queries']))
print('  NDCG@10   :', a1_metrics['NDCG@10'])
print('  MRR       :', a1_metrics['MRR'])
print('  Recall@10 :', a1_metrics['Recall@10'])
print('  Recall@50 :', a1_recall50, '(retrieval ceiling)')

Pipeline A1: MiniLM Dense + QE + Mistral Rerank
Step 1: Dense retrieval (top-50)...
  Recall@50 (retrieval ceiling): 0.815
Step 2: Mistral Large re-ranking (194 queries)...


100%|██████████| 194/194 [16:53<00:00,  5.22s/it]


Pipeline A1 Results (194 queries):
  NDCG@10   : 0.5917
  MRR       : 0.6607
  Recall@10 : 0.6724
  Recall@50 : 0.815 (retrieval ceiling)


## Cell 10 — Pipeline A2: MiniLM Hybrid (α=0.7) + QE + Mistral Rerank
**Hybrid** — α=0.7 * dense_norm + 0.3 * bm25_norm → top-50 → Mistral reranks top-20 → top-10.

In [17]:
print('=' * 60)
print('Pipeline A2: MiniLM Hybrid (alpha={}) + QE + Mistral Rerank'.format(ALPHA))
print('=' * 60)

# Step 1: Hybrid retrieval (dense + BM25 fusion)
print('Step 1: Hybrid retrieval alpha={} (top-{})...'.format(ALPHA, TOP_K))
a2_retrieved = hybrid_retrieve(
    minilm_query_embs, minilm_index, corpus_ids, bm25,
    combined_texts, sample_qids, alpha=ALPHA, top_k=TOP_K
)

eval_ret = pytrec_eval.RelevanceEvaluator(sample_qrels, {'recall_50'})
a2_recall50 = round(np.mean([v['recall_50'] for v in eval_ret.evaluate(a2_retrieved).values()]), 4)
print('  Recall@50 (retrieval ceiling):', a2_recall50)

# Step 2: Mistral Large re-ranking
print('Step 2: Mistral Large re-ranking ({} queries)...'.format(len(sample_qids)))
a2_reranked = {}
for i, (qid, q_text) in enumerate(tqdm(zip(sample_qids, sample_texts), total=len(sample_qids))):
    candidates = list(a2_retrieved[qid].keys())
    a2_reranked[qid] = mistral_rerank(q_text, candidates, corpus_id_to_text, client, top_k=10)
    time.sleep(RATE_DELAY)

# Step 3: Evaluate
a2_metrics = evaluate_metrics(a2_reranked, sample_qrels)
print('\nPipeline A2 Results ({} queries):'.format(a2_metrics['num_queries']))
print('  NDCG@10   :', a2_metrics['NDCG@10'])
print('  MRR       :', a2_metrics['MRR'])
print('  Recall@10 :', a2_metrics['Recall@10'])
print('  Recall@50 :', a2_recall50, '(retrieval ceiling)')

print('\n--- MiniLM pipelines complete ---')
print('A1 NDCG:', a1_metrics['NDCG@10'], '(dense only)')
print('A2 NDCG:', a2_metrics['NDCG@10'], '(hybrid alpha={})'.format(ALPHA))
print('Hybrid delta:', round(a2_metrics['NDCG@10'] - a1_metrics['NDCG@10'], 4))

Pipeline A2: MiniLM Hybrid (alpha=0.7) + QE + Mistral Rerank
Step 1: Hybrid retrieval alpha=0.7 (top-50)...
  Recall@50 (retrieval ceiling): 0.8307
Step 2: Mistral Large re-ranking (194 queries)...


100%|██████████| 194/194 [23:27<00:00,  7.26s/it]   


Pipeline A2 Results (194 queries):
  NDCG@10   : 0.5813
  MRR       : 0.6685
  Recall@10 : 0.6513
  Recall@50 : 0.8307 (retrieval ceiling)

--- MiniLM pipelines complete ---
A1 NDCG: 0.5917 (dense only)
A2 NDCG: 0.5813 (hybrid alpha=0.7)
Hybrid delta: -0.0104


## Cell 11 — Free MiniLM Memory Before Loading BGE-Large

In [12]:
print('Freeing MiniLM memory...')
del minilm
del minilm_corpus_embs
del minilm_index
gc.collect()
print('MiniLM memory freed. Ready to load BGE-Large.')

Freeing MiniLM memory...
MiniLM memory freed. Ready to load BGE-Large.


## Cell 12 — BGE-Large: Encode Corpus + Build FAISS
`BAAI/bge-large-en-v1.5` — 1024-dim, 335M params.  
**BGE query prefix required**: `'Represent this sentence for searching relevant passages: '`  
Corpus is encoded WITHOUT prefix; queries are encoded WITH prefix.

In [11]:
print('=' * 55)
print('Encoding corpus with BGE-Large...')
print('=' * 55)
print('Note: reduced corpus (~6K passages) makes this feasible on CPU')

BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

bge_large = SentenceTransformer('BAAI/bge-large-en-v1.5')
print('  Dim:', bge_large.get_sentence_embedding_dimension(), ' Params: ~335M')

# Encode corpus WITHOUT prefix (BGE convention)
bge_corpus_embs = bge_large.encode(
    corpus_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)

dim_b = bge_corpus_embs.shape[1]
bge_index = faiss.IndexFlatIP(dim_b)
bge_index.add(bge_corpus_embs)
print('  FAISS index: dim={}  vectors={:,}'.format(dim_b, bge_index.ntotal))

# Encode queries WITH prefix + combined QE text
# BGE requires query prefix for correct similarity: encode(prefix + original + expanded)
print('Encoding 30% sample queries with BGE-Large + QE prefix...')
bge_query_texts = [BGE_PREFIX + t for t in combined_texts]
bge_query_embs  = bge_large.encode(
    bge_query_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)

print('BGE-Large ready. Query embs shape:', bge_query_embs.shape)

Encoding corpus with BGE-Large...
Note: reduced corpus (~6K passages) makes this feasible on CPU


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9467.79it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Dim: 1024  Params: ~335M


Batches: 100%|██████████| 97/97 [14:33<00:00,  9.00s/it]


  FAISS index: dim=1024  vectors=6,177
Encoding 30% sample queries with BGE-Large + QE prefix...


Batches: 100%|██████████| 4/4 [00:09<00:00,  2.25s/it]

BGE-Large ready. Query embs shape: (194, 1024)


## Cell 13 — Pipeline B1: BGE-Large Dense + QE + Mistral Rerank
**Dense only** — 1024-dim BGE-Large FAISS search → top-50 → Mistral reranks top-20 → top-10.

In [12]:
print('=' * 60)
print('Pipeline B1: BGE-Large Dense + QE + Mistral Rerank')
print('=' * 60)

# Step 1: Dense retrieval
print('Step 1: Dense retrieval (top-{})...'.format(TOP_K))
b1_retrieved = dense_retrieve(bge_query_embs, bge_index, corpus_ids, sample_qids, top_k=TOP_K)

eval_ret = pytrec_eval.RelevanceEvaluator(sample_qrels, {'recall_50'})
b1_recall50 = round(np.mean([v['recall_50'] for v in eval_ret.evaluate(b1_retrieved).values()]), 4)
print('  Recall@50 (retrieval ceiling):', b1_recall50)

# Step 2: Mistral Large re-ranking
print('Step 2: Mistral Large re-ranking ({} queries)...'.format(len(sample_qids)))
b1_reranked = {}
for i, (qid, q_text) in enumerate(tqdm(zip(sample_qids, sample_texts), total=len(sample_qids))):
    candidates = list(b1_retrieved[qid].keys())
    b1_reranked[qid] = mistral_rerank(q_text, candidates, corpus_id_to_text, client, top_k=10)
    time.sleep(RATE_DELAY)

# Step 3: Evaluate
b1_metrics = evaluate_metrics(b1_reranked, sample_qrels)
print('\nPipeline B1 Results ({} queries):'.format(b1_metrics['num_queries']))
print('  NDCG@10   :', b1_metrics['NDCG@10'])
print('  MRR       :', b1_metrics['MRR'])
print('  Recall@10 :', b1_metrics['Recall@10'])
print('  Recall@50 :', b1_recall50, '(retrieval ceiling)')

Pipeline B1: BGE-Large Dense + QE + Mistral Rerank
Step 1: Dense retrieval (top-50)...
  Recall@50 (retrieval ceiling): 0.8539
Step 2: Mistral Large re-ranking (194 queries)...


100%|██████████| 194/194 [17:14<00:00,  5.33s/it]


Pipeline B1 Results (194 queries):
  NDCG@10   : 0.6056
  MRR       : 0.6679
  Recall@10 : 0.6917
  Recall@50 : 0.8539 (retrieval ceiling)


## Cell 14 — Pipeline B2: BGE-Large Hybrid (α=0.7) + QE + Mistral Rerank
**Hybrid** — 1024-dim BGE-Large + BM25 alpha fusion → top-50 → Mistral reranks top-20 → top-10.

In [13]:
print('=' * 60)
print('Pipeline B2: BGE-Large Hybrid (alpha={}) + QE + Mistral Rerank'.format(ALPHA))
print('=' * 60)

# Step 1: Hybrid retrieval (dense + BM25 fusion, reusing same bm25 object)
print('Step 1: Hybrid retrieval alpha={} (top-{})...'.format(ALPHA, TOP_K))
b2_retrieved = hybrid_retrieve(
    bge_query_embs, bge_index, corpus_ids, bm25,
    combined_texts, sample_qids, alpha=ALPHA, top_k=TOP_K
)

eval_ret = pytrec_eval.RelevanceEvaluator(sample_qrels, {'recall_50'})
b2_recall50 = round(np.mean([v['recall_50'] for v in eval_ret.evaluate(b2_retrieved).values()]), 4)
print('  Recall@50 (retrieval ceiling):', b2_recall50)

# Step 2: Mistral Large re-ranking
print('Step 2: Mistral Large re-ranking ({} queries)...'.format(len(sample_qids)))
b2_reranked = {}
for i, (qid, q_text) in enumerate(tqdm(zip(sample_qids, sample_texts), total=len(sample_qids))):
    candidates = list(b2_retrieved[qid].keys())
    b2_reranked[qid] = mistral_rerank(q_text, candidates, corpus_id_to_text, client, top_k=10)
    time.sleep(RATE_DELAY)

# Step 3: Evaluate
b2_metrics = evaluate_metrics(b2_reranked, sample_qrels)
print('\nPipeline B2 Results ({} queries):'.format(b2_metrics['num_queries']))
print('  NDCG@10   :', b2_metrics['NDCG@10'])
print('  MRR       :', b2_metrics['MRR'])
print('  Recall@10 :', b2_metrics['Recall@10'])
print('  Recall@50 :', b2_recall50, '(retrieval ceiling)')

print('\n--- BGE-Large pipelines complete ---')
print('B1 NDCG:', b1_metrics['NDCG@10'], '(dense only)')
print('B2 NDCG:', b2_metrics['NDCG@10'], '(hybrid alpha={})'.format(ALPHA))
print('Hybrid delta:', round(b2_metrics['NDCG@10'] - b1_metrics['NDCG@10'], 4))

Pipeline B2: BGE-Large Hybrid (alpha=0.7) + QE + Mistral Rerank
Step 1: Hybrid retrieval alpha=0.7 (top-50)...
  Recall@50 (retrieval ceiling): 0.8512
Step 2: Mistral Large re-ranking (194 queries)...


100%|██████████| 194/194 [20:54<00:00,  6.47s/it] 


Pipeline B2 Results (194 queries):
  NDCG@10   : 0.5381
  MRR       : 0.6243
  Recall@10 : 0.5984
  Recall@50 : 0.8512 (retrieval ceiling)

--- BGE-Large pipelines complete ---
B1 NDCG: 0.6056 (dense only)
B2 NDCG: 0.5381 (hybrid alpha=0.7)
Hybrid delta: -0.0675


## Cell 15 — Free BGE-Large Memory

In [ ]:
print('Freeing BGE-Large memory...')
del bge_large
del bge_corpus_embs
del bge_index
gc.collect()
print('BGE-Large memory freed.')

## Cell 16 — Answer Generation + NLI Faithfulness Demo (5 queries)
Runs all 4 pipelines on the same 5 queries for qualitative comparison.  
Shows the full RAG loop: retrieval → rerank → Groq answer → NLI confidence score.

In [18]:
DEMO_QIDS  = sample_qids[:5]  # same 5 queries for all 4 pipelines
DEMO_TEXTS = sample_texts[:5]

pipeline_stores = {
    'A1 (MiniLM Dense)':       {'reranked': a1_reranked, 'retrieved': a1_retrieved},
    'A2 (MiniLM Hybrid α=0.7)':{'reranked': a2_reranked, 'retrieved': a2_retrieved},
    'B1 (BGE-Large Dense)':    {'reranked': b1_reranked, 'retrieved': b1_retrieved},
    'B2 (BGE-Large Hybrid α=0.7)':{'reranked': b2_reranked, 'retrieved': b2_retrieved},
}

demo_summary = []

for pipe_name, pipe_data in pipeline_stores.items():
    print('\n' + '=' * 70)
    print('DEMO —', pipe_name)
    print('=' * 70)
    for qid, q_text in zip(DEMO_QIDS, DEMO_TEXTS):
        top_docs   = list(pipe_data['reranked'][qid].keys())
        answer     = groq_answer(q_text, top_docs, corpus_id_to_text, client)
        time.sleep(RATE_DELAY)
        conf       = compute_confidence(answer, top_docs, corpus_id_to_text,
                                        pipe_data['retrieved'][qid], nli_model)
        fc         = conf['final_confidence']
        label      = ('HIGH' if fc >= 0.7 else ('MEDIUM' if fc >= 0.4 else 'LOW'))

        demo_summary.append({
            'pipeline': pipe_name, 'query_id': qid, 'query': q_text,
            'answer': answer, **conf, 'conf_label': label
        })

        print('\nQuery      :', q_text)
        print('Answer     :', answer[:200], '...')
        print('Retrieval  :', conf['retrieval_confidence'],
              '| Faithfulness:', conf['faithfulness_confidence'],
              '| Final:', fc, '-->', label)

print('\nDemo complete for all 4 pipelines.')


DEMO — A1 (MiniLM Dense)

Query      : What is the easiest way to back-test index funds and ETFs?
Answer     : Based on the provided documents, the easiest way to back-test index funds and ETFs is not explicitly stated. However, Doc 1 mentions a "pastsat-backtesting" tool that allows testing of well-known tech ...
Retrieval  : 0.2866 | Faithfulness: 0.98 | Final: 0.7027 --> HIGH

Query      : Can I deduct interest and fees on a loan for qualified medical expenses?
Answer     : According to Doc 1, "Loan interest and fees do not meet" the definition of medical expenses, and "are a cost of the payment method you chose (a loan), not a cost of medical treatment." Therefore, inte ...
Retrieval  : 0.1408 | Faithfulness: 0.8589 | Final: 0.5716 --> MEDIUM

Query      : Where to find the full book of outstanding bids/asks for a stock?
Answer     : To find the full book of outstanding bids/asks for a stock, you can look at the stock's order book, or level two feed, which shows all the people who

## Cell 17 — Final Comparison Table + Save Results

In [19]:
# ── Reference numbers from previous notebooks ────────────────────────────────
REFERENCE = [
    # (label, NDCG, MRR, Recall10, note)
    ('BM25 Baseline (full corpus)',            0.2169, 0.2706, 0.2784, 'FinChatbot.ipynb'),
    ('MiniLM Dense (full 648 queries)',         0.3687, 0.4451, 0.4413, 'FinChatbot.ipynb'),
    ('MiniLM Hybrid alpha=0.7 (full 648)',      0.3791, 0.4606, 0.4473, 'FinChatbot.ipynb'),
    ('Mistral Rerank top-20 (full 648)',        0.3885, 0.4775, 0.4485, 'FinChatbot.ipynb - BEST'),
]

# ── Current 4-pipeline results (30% sample, reduced corpus) ──────────────────
current_pipelines = [
    ('A1 MiniLM Dense + QE + Mistral',
     a1_metrics['NDCG@10'], a1_metrics['MRR'], a1_metrics['Recall@10'], 'reduced corpus + 30% queries'),
    ('A2 MiniLM Hybrid a=0.7 + QE + Mistral',
     a2_metrics['NDCG@10'], a2_metrics['MRR'], a2_metrics['Recall@10'], 'reduced corpus + 30% queries'),
    ('B1 BGE-Large Dense + QE + Mistral',
     b1_metrics['NDCG@10'], b1_metrics['MRR'], b1_metrics['Recall@10'], 'reduced corpus + 30% queries'),
    ('B2 BGE-Large Hybrid a=0.7 + QE + Mistral',
     b2_metrics['NDCG@10'], b2_metrics['MRR'], b2_metrics['Recall@10'], 'reduced corpus + 30% queries'),
]

all_rows   = REFERENCE + current_pipelines
best_ndcg  = max(r[1] for r in current_pipelines)  # best among 4 new pipelines

print()
print('=' * 95)
print('   4-PIPELINE COMPARISON — MiniLM vs BGE-Large × Dense vs Hybrid (reduced corpus + 30% queries)')
print('=' * 95)
header = '  {:<42} {:>8} {:>8} {:>10}  {}'.format('Pipeline', 'NDCG@10', 'MRR', 'Recall@10', 'Note')
print(header)
print('-' * 95)

print('  [Reference — full corpus results from FinChatbot.ipynb]')
for name, ndcg, mrr, recall, note in REFERENCE:
    row = '  {:<42} {:>8.4f} {:>8.4f} {:>10.4f}  {}'.format(name, ndcg, mrr, recall, note)
    print(row)

print('-' * 95)
print('  [Current — reduced corpus + 30% queries (same random_state=42)]')
for name, ndcg, mrr, recall, note in current_pipelines:
    tag = '  <- BEST' if ndcg == best_ndcg else ''
    row = '  {:<42} {:>8.4f} {:>8.4f} {:>10.4f}  {}{}'.format(name, ndcg, mrr, recall, note, tag)
    print(row)

print('=' * 95)

# ── Observations ─────────────────────────────────────────────────────────────
winner  = max(current_pipelines, key=lambda x: x[1])
a_delta = round(a2_metrics['NDCG@10'] - a1_metrics['NDCG@10'], 4)
b_delta = round(b2_metrics['NDCG@10'] - b1_metrics['NDCG@10'], 4)
ab_dense  = round(b1_metrics['NDCG@10'] - a1_metrics['NDCG@10'], 4)
ab_hybrid = round(b2_metrics['NDCG@10'] - a2_metrics['NDCG@10'], 4)

print()
print('Observations:')
print('  MiniLM:    Hybrid vs Dense delta = {} (+ means hybrid helps)'.format(a_delta))
print('  BGE-Large: Hybrid vs Dense delta = {} (+ means hybrid helps)'.format(b_delta))
print('  Dense:     BGE-Large vs MiniLM   = {} (+ means bigger model helps)'.format(ab_dense))
print('  Hybrid:    BGE-Large vs MiniLM   = {} (+ means bigger model helps)'.format(ab_hybrid))
print()
print('Winner (4 new pipelines) :', winner[0])
print('Winner NDCG@10           :', winner[1])
print()
print('Note: reduced corpus inflates absolute metrics vs full corpus.')
print('Relative ranking between A1/A2/B1/B2 is meaningful for model selection.')

# ── Save results JSON ─────────────────────────────────────────────────────────
out = {
    'evaluation': 'FiQA reduced corpus + 30% sample (random_state=42)',
    'corpus': 'Stratified: all qrel docs + 8% random negatives',
    'config': {
        'TOP_K': TOP_K, 'ALPHA': ALPHA,
        'BM25_K1': BM25_K1, 'BM25_B': BM25_B,
        'reranker': MISTRAL_MODEL, 'answer_model': GROQ_MODEL,
    },
    'pipelines': {
        'A1_MiniLM_Dense':       {k: a1_metrics[k] for k in ['NDCG@10','MRR','Recall@10','num_queries']},
        'A2_MiniLM_Hybrid_0.7':  {k: a2_metrics[k] for k in ['NDCG@10','MRR','Recall@10','num_queries']},
        'B1_BGELarge_Dense':     {k: b1_metrics[k] for k in ['NDCG@10','MRR','Recall@10','num_queries']},
        'B2_BGELarge_Hybrid_0.7':{k: b2_metrics[k] for k in ['NDCG@10','MRR','Recall@10','num_queries']},
    },
    'winner': winner[0],
    'hybrid_delta_MiniLM': a_delta,
    'hybrid_delta_BGELarge': b_delta,
    'reference_full_corpus': {
        name: {'NDCG@10': ndcg, 'MRR': mrr, 'Recall@10': recall}
        for name, ndcg, mrr, recall, _ in REFERENCE
    },
}
out_path = os.path.join(PIPELINE_DIR, 'pipeline_comparison.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print('Saved:', out_path)


   4-PIPELINE COMPARISON — MiniLM vs BGE-Large × Dense vs Hybrid (reduced corpus + 30% queries)
  Pipeline                                    NDCG@10      MRR  Recall@10  Note
-----------------------------------------------------------------------------------------------
  [Reference — full corpus results from FinChatbot.ipynb]
  BM25 Baseline (full corpus)                  0.2169   0.2706     0.2784  FinChatbot.ipynb
  MiniLM Dense (full 648 queries)              0.3687   0.4451     0.4413  FinChatbot.ipynb
  MiniLM Hybrid alpha=0.7 (full 648)           0.3791   0.4606     0.4473  FinChatbot.ipynb
  Mistral Rerank top-20 (full 648)             0.3885   0.4775     0.4485  FinChatbot.ipynb - BEST
-----------------------------------------------------------------------------------------------
  [Current — reduced corpus + 30% queries (same random_state=42)]
  A1 MiniLM Dense + QE + Mistral               0.5917   0.6607     0.6724  reduced corpus + 30% queries
  A2 MiniLM Hybrid a=0.7 + Q